# Contrail Labeling Helper

Interactive tool for labeling flight contrails.

## Labeling Protocol
- **Blocked**: Flight path obscured (clouds, etc.)
- **Clear**: No contrail visible
- **Dissipate < 10**: Contrail dissipates in < 10 seconds
- **Dissipate > 10**: Contrail dissipates in > 10 seconds
- **Persistent**: Contrail persists for extended time

In [15]:
import pandas as pd
import numpy as np
import cv2
import os
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import json

import utils.adsb_utils as adsb_utils
import utils.projection_utils as proj_utils
from utils.image_data_utils import get_image_data_uwisc
# reload import


In [58]:
import importlib
importlib.reload(adsb_utils)
importlib.reload(proj_utils)

<module 'utils.projection_utils' from '/Users/shrenikborad/pless/contrails/utils/projection_utils.py'>

## Configuration
Set your date, camera side, and paths below:

In [75]:
# Configuration
DATE_STR = "2025-01-09"  # Change this to your date
CAMERA_SIDE = "east"     # 'east' or 'south'

# Paths
ADSB_CSV_PATH = f"/Users/shrenikborad/pless/contrails/adsb_flightpings_wisconsin_2025-01-09.csv"
CAMERA_PARAMS_PATH = f"./uwisc/{CAMERA_SIDE}/camera_params.json"
BASE_DIR = f'/Users/shrenikborad/Downloads/NNDL/images_uwisc/{CAMERA_SIDE}/{DATE_STR}/{CAMERA_SIDE}'
CAMERA_NAME = f"uwisc_{CAMERA_SIDE}"

# Output path for labels
LABELS_OUTPUT_PATH = f"./contrail_labels_{DATE_STR}_{CAMERA_SIDE}.csv"

print(f"Date: {DATE_STR}")
print(f"Camera: {CAMERA_SIDE}")
print(f"Labels will be saved to: {LABELS_OUTPUT_PATH}")

Date: 2025-01-09
Camera: east
Labels will be saved to: ./contrail_labels_2025-01-09_east.csv


## Load Data

In [ ]:
# Load ADSB data
print("Loading ADSB data...")
df = pd.read_csv(ADSB_CSV_PATH)

# Filter to daytime hours
from_dt = pd.to_datetime(f"{DATE_STR} 06:00:00").tz_localize('America/Chicago').tz_convert('UTC')
to_dt = pd.to_datetime(f"{DATE_STR} 19:00:00").tz_localize('America/Chicago').tz_convert('UTC')
df['time'] = pd.to_datetime(df['time'])
df = df[(df['time'] >= from_dt) & (df['time'] < to_dt)]

print(f"Loaded {len(df)} ADSB pings")

# Upsample flight data
print("Upsampling flight data...")
df_upsampled = adsb_utils.get_upsampled_df_for_day(df, max_range_m=100000)

# Load camera parameters and project to image coordinates
print("Projecting to image coordinates...")
intrinsics, distortion, rvec, tvec, origin_gps = proj_utils.load_camera_parameters(CAMERA_PARAMS_PATH)

image_x, image_y, cam_distance = proj_utils.gps_to_camxy_vasha_fixed(
    df_upsampled['lat'].values,
    df_upsampled['lon'].values,
    df_upsampled['alt_gnss_meters'].values,
    cam_k=intrinsics,
    cam_r=rvec,
    cam_t=tvec,
    camera_gps=origin_gps,
    distortion=distortion
)

df_upsampled['image_x'] = image_x
df_upsampled['image_y'] = image_y
df_upsampled['cam_distance'] = cam_distance

# Load image metadata
print("Loading image metadata...")
image_df = get_image_data_uwisc(BASE_DIR, DATE_STR)
image_df = image_df[(image_df['time'] >= from_dt) & (image_df['time'] < to_dt)]
max_time = image_df['time'].max() + timedelta(seconds=1)
min_time = image_df['time'].min() - timedelta(seconds=1)
image_df = image_df.sort_values('time').reset_index(drop=True)
df_upsampled = df_upsampled[
    (df_upsampled['time'] >= min_time) & 
    (df_upsampled['time'] <= max_time)
].copy()

print(f"Loaded {len(image_df)} images")
print(f"Time range: {image_df['time'].min()} to {image_df['time'].max()}")

Loading ADSB data...
Loaded 5259 ADSB pings
Upsampling flight data...
Upsampling all aircraft...
Processing 254 unique aircraft...



/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time'] = pd.to_datetime(df['time'])
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['alt_gnss_meters'] = df['alt_gnss_meters'].astype(float)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:111: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Processed 10 aircraft...
Processed 20 aircraft...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_df = pd.concat([interp_df, interp_points], ignore_index=True)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:82: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df[col] = temp_df[col].fillna(method='ffill')
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before

Processed 30 aircraft...
Processed 40 aircraft...


...

/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_df = pd.concat([interp_df, interp_points], ignore_index=True)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:82: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df[col] = temp_df[col].fillna(method='ffill')
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before

Processed 240 aircraft...
Processed 250 aircraft...


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:82: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df[col] = temp_df[col].fillna(method='ffill')
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_df = pd.concat([interp_df, interp_points], ignore_index=True)
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:82: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  temp_df[col] = temp_df[col].fillna(method='ffill')
/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:64: FutureWarning: The behavi

Projecting to image coordinates...
Loading image metadata...
Total images between 3 pm and 4 pm utc: 3960
                       time            image_file
0 2025-01-09 14:00:08+00:00  08_00_08.trig+00.jpg
1 2025-01-09 14:00:18+00:00  08_00_18.trig+00.jpg
2 2025-01-09 14:00:28+00:00  08_00_28.trig+00.jpg
3 2025-01-09 14:00:38+00:00  08_00_38.trig+00.jpg
4 2025-01-09 14:00:48+00:00  08_00_48.trig+00.jpg
Loaded 3960 images
Time range: 2025-01-09 14:00:08+00:00 to 2025-01-10 00:59:58+00:00


/Users/shrenikborad/pless/contrails/utils/adsb_utils.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_upsampled[['lat', 'lon', 'alt_gnss_meters']].applymap(lambda x: isinstance(x, str)).any(axis=1)
/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: divide by zero encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t.T).T  # Shape: (N, 3)
/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: overflow encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t.T).T  # Shape: (N, 3)
/Users/shrenikborad/pless/contrails/utils/projection_utils.py:33: RuntimeWarning: invalid value encountered in matmul
  points_cam = (cam_r @ enu_points.T + cam_t.T).T  # Shape: (N, 3)


In [77]:
# Get unique flights visible in the time window
# A flight is visible if it has valid image coordinates
image = image_df.iloc[0]
cv2_image = cv2.imread(BASE_DIR + "/"+ image['image_file'])
image_height, image_width = cv2_image.shape[:2]
df_visible = df_upsampled[
    (df_upsampled['image_x'].notna()) & 
    (df_upsampled['image_y'].notna()) &
    (df_upsampled['image_x'] >= 0) &
    (df_upsampled['image_y'] >= 0) &
    (df_upsampled['image_x'] < image_width) &
    (df_upsampled['image_y'] < image_height)
].copy()

# Get flight summary
flight_summary = df_visible.groupby('ident').agg({
    'time': ['min', 'max', 'count'],
    'alt_gnss_meters': ['min', 'max', 'mean'],
    'image_x': 'mean',
    'image_y': 'mean'
}).reset_index()

flight_summary.columns = ['ident', 'time_appear', 'time_disappear', 'n_points', 
                          'alt_min', 'alt_max', 'alt_mean', 'avg_x', 'avg_y']

# Convert altitude to feet for display
flight_summary['alt_min_ft'] = (flight_summary['alt_min'] * 3.28084).round(0)
flight_summary['alt_max_ft'] = (flight_summary['alt_max'] * 3.28084).round(0)

print(f"Found {len(flight_summary)} unique flights visible in frame")
flight_summary.head(10)

Found 172 unique flights visible in frame


,ident,time_appear,time_disappear,n_points,alt_min,alt_max,alt_mean,avg_x,avg_y,alt_min_ft,alt_max_ft
0,11,2025-01-09 19:02:22+00:00,2025-01-09 19:05:35+00:00,194,11445.24,11445.240000,11445.240000,1389.659221,627.615346,37550.0,37550.0
1,AAL1057,2025-01-09 20:02:21+00:00,2025-01-09 20:05:16+00:00,176,9669.78,9677.400000,9674.347670,1687.131644,629.326687,31725.0,31750.0
2,AAL1964,2025-01-09 14:03:45+00:00,2025-01-09 14:06:19+00:00,155,6728.46,8101.654884,7473.679422,2004.278616,856.705214,22075.0,26580.0
3,AAL2222,2025-01-09 18:17:47+00:00,2025-01-09 18:18:47+00:00,51,10850.88,10850.880000,10850.880000,101.605416,809.989976,35600.0,35600.0
4,AAL2455,2025-01-09 15:36:06+00:00,2025-01-09 15:39:34+00:00,190,10248.90,10256.520000,10252.572305,1070.372743,653.007038,33625.0,33650.0
5,AAL3099,2025-01-09 14:17:46+00:00,2025-01-09 14:18:40+00:00,55,11475.72,11475.720000,11475.720000,167.212209,809.187464,37650.0,37650.0
6,AAL667,2025-01-09 19:43:35+00:00,2025-01-09 19:45:07+00:00,77,3375.66,3939.540000,3604.556883,2440.017166,1064.830580,11075.0,12925.0
7,ACA1039,2025-01-09 15:14:58+00:00,2025-01-09 15:18:27+00:00,210,10843.26,10846.308000,10843.354343,1268.664293,668.759630,35575.0,35585.0
8,ACA1041,2025-01-09 20:58:00+00:00,2025-01-09 20:59:22+00:00,83,10828.02,10828.020000,10828.020000,126.747381,752.348017,35525.0,35525.0
9,ACA1073,2025-01-09 15:14:04+00:00,2025-01-09 15:16:33+00:00,150,9654.54,9654.540000,9654.540000,1153.215510,613.669207,31675.0,31675.0


## Initialize Labels DataFrame

In [78]:
# Initialize or load existing labels
LABEL_OPTIONS = ['', 'Blocked', 'Clear', 'Dissipate < 10', 'Dissipate > 10', 'Persistent']

labels_df = flight_summary[['ident', 'time_appear', 'time_disappear', 
                                'alt_min', 'alt_max', 'alt_min_ft', 'alt_max_ft']].copy()
labels_df['alt_appear'] = labels_df['alt_min']
labels_df['alt_disappear'] = labels_df['alt_max']
labels_df['label'] = ''
labels_df['notes'] = ''
labels_df['section'] = 1  # For tracking splits

print(f"Labels dataframe has {len(labels_df)} entries")
labels_df.head()

Labels dataframe has 172 entries


,ident,time_appear,time_disappear,alt_min,alt_max,alt_min_ft,alt_max_ft,alt_appear,alt_disappear,label,notes,section
0,11,2025-01-09 19:02:22+00:00,2025-01-09 19:05:35+00:00,11445.24,11445.240000,37550.0,37550.0,11445.24,11445.240000,,,1
1,AAL1057,2025-01-09 20:02:21+00:00,2025-01-09 20:05:16+00:00,9669.78,9677.400000,31725.0,31750.0,9669.78,9677.400000,,,1
2,AAL1964,2025-01-09 14:03:45+00:00,2025-01-09 14:06:19+00:00,6728.46,8101.654884,22075.0,26580.0,6728.46,8101.654884,,,1
3,AAL2222,2025-01-09 18:17:47+00:00,2025-01-09 18:18:47+00:00,10850.88,10850.880000,35600.0,35600.0,10850.88,10850.880000,,,1
4,AAL2455,2025-01-09 15:36:06+00:00,2025-01-09 15:39:34+00:00,10248.90,10256.520000,33625.0,33650.0,10248.90,10256.520000,,,1


## Interactive Labeling Interface

In [79]:
# Export data for HTML labeler
import json

def export_labeling_data(image_df, df_upsampled, labels_df, base_dir, output_json_path):
    """Export all data needed for the HTML labeling interface."""
    
    frames = []
    for idx, row in image_df.iterrows():
        t = row['time']
        
        # Get flights at this time
        flights_at_time = df_upsampled[df_upsampled['time'] == t].copy()
        flights_at_time = flights_at_time[
            (flights_at_time['image_x'].notna()) & 
            (flights_at_time['image_y'].notna()) &
            (flights_at_time['image_x'] >= 0) &
            (flights_at_time['image_y'] >= 0) &
            (flights_at_time['image_x'] < 3000) &  # Filter unrealistic values
            (flights_at_time['image_y'] < 3000)
        ]
        
        flights_list = []
        for _, f in flights_at_time.iterrows():
            flights_list.append({
                'ident': f['ident'],
                'x': float(f['image_x']),
                'y': float(f['image_y']),
                'alt_ft': round(f['alt_gnss_meters'] * 3.28084),
                'heading': float(f['heading']) if 'heading' in f and pd.notna(f['heading']) else 0
            })
        
        frames.append({
            'idx': idx,
            'image_file': row['image_file'],
            'time_utc': t.strftime('%Y-%m-%d %H:%M:%S'),
            'time_local': t.tz_convert('America/Chicago').strftime('%H:%M:%S'),
            'flights': flights_list
        })
    
    # Flight summary for labels
    flights_in_frames = set()
    for frame in frames:
        for flight in frame['flights']:
            flights_in_frames.add(flight['ident'])
    flight_list = []
    for _, row in labels_df.iterrows():
        if row['ident'] not in flights_in_frames:
            continue
        flight_list.append({
            'ident': row['ident'],
            'time_appear': pd.to_datetime(row['time_appear']).strftime('%H:%M:%S'),
            'time_disappear': pd.to_datetime(row['time_disappear']).strftime('%H:%M:%S'),
            'alt_min_ft': int(row['alt_min_ft']),
            'alt_max_ft': int(row['alt_max_ft']),
            'label': row['label'] if pd.notna(row['label']) else '',
            'notes': row['notes'] if pd.notna(row['notes']) else '',
            'section': int(row.get('section', 1))
        })
    
    data = {
        'date': DATE_STR,
        'camera': CAMERA_SIDE,
        'image_base_path': base_dir,
        'total_frames': len(frames),
        'frames': frames,
        'flights': flight_list
    }
    
    with open(output_json_path, 'w') as f:
        json.dump(data, f)
    
    print(f"Exported {len(frames)} frames with flight data to {output_json_path}")
    return data

# Export the data
labeling_data = export_labeling_data(
    image_df, df_upsampled, labels_df, BASE_DIR,
    f"./labeling_data_{DATE_STR}_{CAMERA_SIDE}.json"
)

Exported 3960 frames with flight data to ./labeling_data_2025-01-09_east.json
